# Fusionformer: Full Dissertation Codebase

**Author:** Nachiket Magadum (2550458)
**Institution:** Brunel University London, MSc AI
**Supervisor:** Prof Xiaohui Liu
**Submission:** September 2026

This single notebook stitches every module of the Fusionformer reproduction and ablation study into one runnable file. The notebook mirrors the structure of the source repository:

1. Setup and dependencies
2. Data loaders (SKAB, SMD)
3. Evaluation harness (AUROC, PR-AUC, F1-PA, Event-F1)
4. Model implementations (FAM, TimeOnly, SWSE, adversarial)
5. Baseline models (Isolation Forest, LSTM autoencoder)
6. Training loops (SKAB and SMD)
7. Ablation experiments (FAM vs TimeOnly, SWSE sweep, full paper)
8. Statistical significance (Wilcoxon signed-rank + Rosenthal effect size)
9. F1 telemetry cross-domain case study
10. Plotting utilities

All results reported in Chapter 4 of the dissertation are reproducible by running the cells below in order, given SKAB and SMD are placed under `datasets/`.

---

## Prerequisites

- Python 3.10+
- PyTorch (with MPS on Apple Silicon or CUDA on NVIDIA)
- scikit-learn, scipy, numpy, pandas, matplotlib
- fastf1 (for Section 9 only)

Datasets:
- SKAB: place under `datasets/SKAB/data/` (three folders: valve1, valve2, other)
- SMD: place under `datasets/SMD/` (train, test, test_label, interpretation_label)

Both datasets are freely available from their respective public sources.


## Section 1: Setup and Dependencies


In [ ]:
# Standard library
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, precision_score, recall_score
from scipy import stats

# Deep learning
import torch
import torch.nn as nn

# Fix seeds for reproducibility (matches dissertation experiments)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device auto-select
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")


## Section 2: Data Loaders

### 2.1 SKAB Loader

SKAB (Skoltech Anomaly Benchmark) provides 23 CSVs of industrial pump telemetry across three fault scenarios (valve1, valve2, other) with binary anomaly labels.

In [ ]:
from pathlib import Path
import pandas as pd

SKAB_ROOT = Path("datasets/SKAB")

def list_skab_files():
    """Return list of all 23 SKAB CSV file paths."""
    root = SKAB_ROOT / "data"
    if not root.exists():
        raise FileNotFoundError(f"SKAB not found at {root}")
    return sorted(root.glob("*/*.csv"))

def load_skab_file(path):
    """
    Load a single SKAB CSV. Returns (X, y).

    X: DataFrame of 8 sensor features (accelerometers, current, pressure,
       temperature, thermocouple, voltage, volume flow rate).
    y: Series of binary anomaly labels (0=normal, 1=anomaly).
    """
    df = pd.read_csv(path, sep=';', parse_dates=['datetime'], index_col='datetime')
    # SKAB has an 'anomaly' column (0/1) and a 'changepoint' column (0/1).
    # We use 'anomaly' as the ground-truth label.
    y = df['anomaly'].astype(int)
    feature_cols = [c for c in df.columns if c not in ('anomaly', 'changepoint')]
    X = df[feature_cols]
    return X, y

# Quick sanity check (uncomment if datasets available)
# files = list_skab_files()
# print(f"Found {len(files)} SKAB files")
# X, y = load_skab_file(files[0])
# print(f"First file: {X.shape}, anomaly rate {y.mean():.1%}")


### 2.2 SMD Loader

SMD (Server Machine Dataset) provides 28 machines of server telemetry across three groups with 38 features per machine.

In [ ]:
"""
SMD (Server Machine Dataset) loader.

Provides a load_smd_file() function that mirrors the SKAB loader signature:
    X, y = load_smd_file(machine_name)

Where X is a pandas DataFrame of features and y is a pandas Series of binary
anomaly labels.

SMD structure:
    datasets/SMD/train/{machine}.txt         — training data (all normal)
    datasets/SMD/test/{machine}.txt          — test data (contains anomalies)
    datasets/SMD/test_label/{machine}.txt    — binary labels for test
    datasets/SMD/interpretation_label/{machine}.txt — which features are anomalous

The loader concatenates train + test into one sequence so the existing
semi-supervised training pipeline (train on pre-anomaly region) works
without modification.

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""

from pathlib import Path
import numpy as np
import pandas as pd


SMD_ROOT = Path("datasets/SMD")


def list_smd_machines() -> list:
    """Return sorted list of available machine names, e.g. 'machine-1-1'."""
    train_dir = SMD_ROOT / "train"
    if not train_dir.exists():
        raise FileNotFoundError(f"SMD train folder not found at {train_dir}")
    files = sorted(train_dir.glob("*.txt"))
    return [f.stem for f in files]


def load_smd_file(machine: str) -> tuple:
    """
    Load one SMD machine's data.

    Args:
        machine: machine name, e.g. 'machine-1-1'.

    Returns:
        X: pandas DataFrame of shape (n_rows, 38) with feature columns.
        y: pandas Series of shape (n_rows,) with binary anomaly labels
           (0 = normal, 1 = anomaly). Train portion is all zeros.
    """
    train_path = SMD_ROOT / "train" / f"{machine}.txt"
    test_path = SMD_ROOT / "test" / f"{machine}.txt"
    label_path = SMD_ROOT / "test_label" / f"{machine}.txt"

    for p in (train_path, test_path, label_path):
        if not p.exists():
            raise FileNotFoundError(f"SMD file not found: {p}")

    # Load train + test features.
    X_train = pd.read_csv(train_path, header=None)
    X_test = pd.read_csv(test_path, header=None)
    y_test = pd.read_csv(label_path, header=None).iloc[:, 0]

    # Sanity check: same number of columns in train and test.
    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError(
            f"Feature count mismatch: train has {X_train.shape[1]}, "
            f"test has {X_test.shape[1]}"
        )

    # Sanity check: test rows match label rows.
    if len(X_test) != len(y_test):
        raise ValueError(
            f"Test rows ({len(X_test)}) does not match label rows ({len(y_test)})"
        )

    # Give columns names f0..f37 to mirror how SKAB features look.
    feature_names = [f"f{i}" for i in range(X_train.shape[1])]
    X_train.columns = feature_names
    X_test.columns = feature_names

    # Concatenate train + test into one sequence.
    X = pd.concat([X_train, X_test], axis=0, ignore_index=True)

    # Labels: 0 for all train rows, real labels for test rows.
    y_train = pd.Series(np.zeros(len(X_train), dtype=int))
    y = pd.concat([y_train, y_test], axis=0, ignore_index=True)

    return X, y


if __name__ == "__main__":
    # Quick sanity check.
    machines = list_smd_machines()
    print(f"Found {len(machines)} SMD machines: {machines[:5]}...")

    m = machines[0]
    X, y = load_smd_file(m)
    print(f"\nLoaded {m}:")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Feature range: {X.min().min():.2f} to {X.max().max():.2f}")
    print(f"  Total anomalies: {int(y.sum())} / {len(y)} ({y.mean() * 100:.1f}%)")
    first_anom = int(np.argmax(y.values == 1)) if y.any() else len(y)
    print(f"  First anomaly index: {first_anom} (train ends at {len(X) - len(y[y.index >= 0]) if False else 'end of train portion'})")


## Section 3: Evaluation Harness

Computes AUROC, PR-AUC, F1-PA, and Event-F1 following Wu and Keogh (2021) recommendations. The top-k threshold rule for Event-F1 avoids the point-adjustment inflation identified in that paper.

In [ ]:
def compute_all_metrics(y_true, scores):
    """
    Compute AUROC, PR-AUC, F1-PA (inflated - reported for compatibility only),
    and Event-F1 (honest per-window F1 with top-k threshold).

    Args:
        y_true: 1D array of binary anomaly labels.
        scores: 1D array of anomaly scores (higher = more anomalous).

    Returns:
        Dict with keys 'auroc', 'pr_auc', 'f1_pa', 'event_f1', 'k_used', 'threshold'.
    """
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)

    # AUROC (threshold-independent rank quality)
    auroc = roc_auc_score(y_true, scores) if y_true.sum() > 0 else 0.5

    # PR-AUC (class-imbalance robust)
    precision, recall, _ = precision_recall_curve(y_true, scores)
    pr_auc = auc(recall, precision)

    # Top-k threshold: pick k = number of true anomalies
    k = int(y_true.sum())
    if k == 0 or k >= len(y_true):
        threshold = np.median(scores)
        y_pred = (scores > threshold).astype(int)
    else:
        threshold = np.sort(scores)[-k]
        y_pred = (scores >= threshold).astype(int)

    # Event-F1: standard per-point F1 with top-k prediction
    event_f1 = f1_score(y_true, y_pred, zero_division=0)

    # F1-PA: point-adjusted F1 (INFLATED - reported only for compatibility with older literature)
    y_pred_pa = y_pred.copy()
    # Simple contiguous segment adjustment
    in_anomaly = False
    seg_start = 0
    for i in range(len(y_true)):
        if y_true[i] == 1 and not in_anomaly:
            in_anomaly = True
            seg_start = i
        elif y_true[i] == 0 and in_anomaly:
            if y_pred[seg_start:i].any():
                y_pred_pa[seg_start:i] = 1
            in_anomaly = False
    if in_anomaly and y_pred[seg_start:].any():
        y_pred_pa[seg_start:] = 1
    f1_pa = f1_score(y_true, y_pred_pa, zero_division=0)

    return {
        'auroc': auroc, 'pr_auc': pr_auc,
        'f1_pa': f1_pa, 'event_f1': event_f1,
        'k_used': k, 'threshold': threshold,
    }


## Section 4: Fusionformer Model Implementations

### 4.1 FusionAttention (Dual-Axis)

Combines time-axis and channel-axis self-attention with learnable fusion weights.

In [ ]:
"""
Fusion Attention Module for Multivariate Time-Series Anomaly Detection.

Core attention mechanism of the Fusionformer architecture.
Combines attention along two axes:

  1. Time axis  — captures dependencies across time steps.
  2. Channel axis — captures relationships between sensor channels.

The two attention outputs are fused via a learnable weighted sum, letting
the model adaptively balance temporal versus cross-channel information.

Why fusion attention?
    Standard transformer attention operates on one axis only (typically time).
    For industrial multivariate sensor data, both axes matter: a fault might
    manifest as an unusual temporal pattern OR as an unusual relationship
    between sensors. Fusion attention captures both.

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""
import torch
import torch.nn as nn


class FusionAttention(nn.Module):
    """
    Dual-axis attention over multivariate time-series.

    Given input of shape (batch, seq_len, n_features), this module:
        1. Projects features into a d_model latent space.
        2. Applies self-attention across the time axis.
        3. Applies self-attention across the channel axis (via transpose).
        4. Fuses the two outputs with learnable weights.
        5. Projects back to original feature dimension.

    Args:
        seq_len:    Length of the input time window (number of time steps).
        n_features: Number of sensor channels in the input.
        d_model:    Internal latent dimension for projections. Larger = more
                    capacity but slower training. Default 32 works well for
                    industrial sensor tasks with ~10 channels.
    """

    def __init__(self, seq_len: int, n_features: int, d_model: int = 32):
        super().__init__()

        # Store shape parameters for downstream inspection / debugging.
        self.seq_len = seq_len
        self.n_features = n_features
        self.d_model = d_model

        # Input projection: lifts raw features into a richer latent space.
        # Without this, attention would be too narrow at just n_features dims.
        self.input_projection = nn.Linear(n_features, d_model)

        # Time-axis attention: each time step attends to every other time step.
        # embed_dim = d_model because tokens here are time steps in latent space.
        # num_heads = 1 for interpretability in the baseline; can be raised later.
        self.time_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=1,
            batch_first=True,
        )

        # Channel-axis attention: each channel attends to every other channel.
        # embed_dim = seq_len because tokens are channels, each represented
        # by their values across all time steps.
        self.channel_attention = nn.MultiheadAttention(
            embed_dim=seq_len,
            num_heads=1,
            batch_first=True,
        )

        # Learnable weights to fuse time-attention and channel-attention outputs.
        # Initialised equal (0.5, 0.5), then softmaxed at forward time so both
        # weights stay positive and sum to 1 during training.
        self.fusion_weights = nn.Parameter(torch.tensor([0.5, 0.5]))

        # Output projection: maps fused latent back to original feature space.
        # Lets us plug this module in place of a standard attention block
        # without changing downstream dimensions.
        self.output_projection = nn.Linear(d_model, n_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch, seq_len, n_features).

        Returns:
            Tensor of shape (batch, seq_len, n_features).
        """
        # Lift features into latent space:  (B, T, F) -> (B, T, d_model).
        h = self.input_projection(x)

        # Time-axis self-attention. Queries, keys, values all = h.
        # Attention weights implicitly have shape (B, T, T).
        time_out, _ = self.time_attention(h, h, h)

        # Transpose so CHANNELS become the token axis:
        # (B, T, d_model) -> (B, d_model, T).
        # Each of the d_model "channels" now attends across all time steps.
        h_t = h.transpose(1, 2)

        # Channel-axis self-attention. Output has same shape as h_t.
        channel_out, _ = self.channel_attention(h_t, h_t, h_t)

        # Transpose back so shape matches time_out:
        # (B, d_model, T) -> (B, T, d_model).
        channel_out = channel_out.transpose(1, 2)

        # Softmax fusion weights so they behave like a probability distribution.
        # Keeps the fused output on the same scale as its inputs.
        weights = torch.softmax(self.fusion_weights, dim=0)

        # Weighted combination of the two attention outputs.
        fused = weights[0] * time_out + weights[1] * channel_out

        # Project back to feature space: (B, T, d_model) -> (B, T, n_features).
        return self.output_projection(fused)


### 4.2 TransformerBlock

Wraps FusionAttention with residual connections, LayerNorm, and a position-wise feed-forward network (post-norm).

In [ ]:
"""
Transformer Block for the Fusionformer architecture.

Wraps FusionAttention into a standard transformer block with:
    - Residual connections around attention and feed-forward layers
    - Layer normalisation after each residual add
    - A position-wise feed-forward network (MLP)
    - Dropout on both sub-layer outputs

This follows the classic post-norm transformer pattern from Vaswani et al. 2017,
but with FusionAttention replacing standard multi-head self-attention. The block
is the fundamental unit that will be stacked N times inside the encoder and
decoder halves of Fusionformer.

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""
import torch
import torch.nn as nn

# (import from earlier cell)


class TransformerBlock(nn.Module):
    """
    A single transformer block using FusionAttention.

    Given input of shape (batch, seq_len, n_features), applies:
        1. FusionAttention (dual-axis) with residual + LayerNorm.
        2. Position-wise feed-forward network with residual + LayerNorm.

    Args:
        seq_len:    Length of the input time window.
        n_features: Number of channels in the input.
        d_model:    Internal latent dim used inside FusionAttention.
        ff_hidden:  Hidden dim of the feed-forward sub-layer. Common practice
                    is 2-4x n_features. Default 64 balances capacity and cost.
        dropout:    Dropout probability applied to both sub-layer outputs.
                    Helps regularisation on limited industrial datasets.
    """

    def __init__(
        self,
        seq_len: int,
        n_features: int,
        d_model: int = 32,
        ff_hidden: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()

        # Attention sub-layer: our dual-axis FusionAttention module.
        self.attention = FusionAttention(seq_len, n_features, d_model)

        # LayerNorm after the attention residual. Normalises across the feature
        # dim only — the standard choice for post-norm transformers.
        self.norm1 = nn.LayerNorm(n_features)

        # Feed-forward sub-layer: a small MLP applied position-wise (i.e., the
        # same MLP is applied independently to every time step). Widens to
        # ff_hidden and squeezes back — classic transformer FFN pattern.
        self.feed_forward = nn.Sequential(
            nn.Linear(n_features, ff_hidden),
            nn.ReLU(),                       # non-linearity so FFN adds capacity
            nn.Dropout(dropout),             # regularise inside the FFN
            nn.Linear(ff_hidden, n_features),
        )

        # LayerNorm after the feed-forward residual.
        self.norm2 = nn.LayerNorm(n_features)

        # Dropout applied to sub-layer outputs before the residual add.
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch, seq_len, n_features).

        Returns:
            Tensor of shape (batch, seq_len, n_features).
        """
        # Sub-layer 1: attention with residual + norm.
        #   x + dropout(attn(x))  →  LayerNorm
        attn_out = self.attention(x)
        x = self.norm1(x + self.dropout(attn_out))

        # Sub-layer 2: feed-forward with residual + norm.
        #   x + dropout(ffn(x))  →  LayerNorm
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x


### 4.3 Encoder

Stacks N TransformerBlocks with learned positional encoding.

In [ ]:
"""
FusionformerEncoder for the Fusionformer autoencoder architecture.

Stacks N TransformerBlocks with a learned positional encoding at the input.
The encoder processes an input window through repeated attention + feed-forward
blocks, producing a representation that the decoder will reconstruct back into
the original space.

Why positional encoding?
    Attention is permutation-invariant by default — it doesn't know that
    time step 5 comes after time step 4. For time-series data where temporal
    order is critical (rising trends, drift, cycles), we inject position
    information so the model can learn temporal patterns.

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""
import torch
import torch.nn as nn

# (import from earlier cell)


class FusionformerEncoder(nn.Module):
    """
    Stack of N TransformerBlocks with learned positional encoding.

    Args:
        seq_len:    Length of input time window (number of time steps).
        n_features: Number of sensor channels.
        d_model:    Latent dim used inside each block's attention.
        ff_hidden:  Hidden dim of each block's feed-forward layer.
        n_layers:   How many TransformerBlocks to stack.
        dropout:    Dropout rate used inside blocks and on positional encoding.
    """

    def __init__(
        self,
        seq_len: int,
        n_features: int,
        d_model: int = 32,
        ff_hidden: int = 64,
        n_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.n_layers = n_layers

        # Learned positional encoding — one vector per time step per feature.
        # Small initialisation (std 0.02) so it doesn't dominate the input signal
        # early in training. Same trick used by BERT and other modern transformers.
        # Shape: (1, seq_len, n_features) so it broadcasts across the batch dim.
        self.pos_encoding = nn.Parameter(
            torch.randn(1, seq_len, n_features) * 0.02
        )

        # Dropout on the input embedding + positional encoding sum.
        # Helps regularise on limited industrial datasets like SKAB.
        self.dropout = nn.Dropout(dropout)

        # Stack of N TransformerBlocks. nn.ModuleList registers each block
        # as a submodule so all parameters are tracked by the optimiser.
        self.blocks = nn.ModuleList([
            TransformerBlock(
                seq_len=seq_len,
                n_features=n_features,
                d_model=d_model,
                ff_hidden=ff_hidden,
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch, seq_len, n_features).

        Returns:
            Tensor of shape (batch, seq_len, n_features).
        """
        # Add positional information to the input.
        # Broadcasting: pos_encoding is (1, T, F), x is (B, T, F).
        x = x + self.pos_encoding

        # Dropout on the input embedding for regularisation.
        x = self.dropout(x)

        # Pass through each transformer block sequentially.
        for block in self.blocks:
            x = block(x)

        return x


### 4.4 Decoder

Mirror stack of N TransformerBlocks.

In [ ]:
"""
FusionformerDecoder for the Fusionformer autoencoder architecture.

Mirror structure to FusionformerEncoder — stacks N TransformerBlocks to
reconstruct the input from the encoded representation.

Design choices:
    - No positional encoding needed here. The encoder's output already carries
      the temporal information injected upstream.
    - Same TransformerBlock as the encoder, keeping the two halves symmetric
      and parameter-counts easy to reason about.
    - No explicit bottleneck. Reconstruction pressure through the MSE loss
      forces the model to learn meaningful representations, which is the
      standard "flat autoencoder" pattern used for anomaly detection.

Why a flat autoencoder rather than a dimensionality bottleneck?
    Industrial sensor windows are short (typically 30-60 time steps × ~10
    channels). A hard bottleneck at these small dimensions risks throwing
    away useful signal. The reconstruction objective alone provides enough
    pressure to learn a normal-behaviour representation; anomalies then
    fail to reconstruct cleanly and are flagged by their reconstruction
    error.

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""
import torch
import torch.nn as nn

# (import from earlier cell)


class FusionformerDecoder(nn.Module):
    """
    Stack of N TransformerBlocks that reconstructs the input from encoded features.

    Args:
        seq_len:    Length of input time window.
        n_features: Number of sensor channels.
        d_model:    Latent dim inside each block's attention.
        ff_hidden:  Hidden dim of each block's feed-forward layer.
        n_layers:   How many TransformerBlocks to stack. Keep same as encoder
                    by default so the model is symmetric.
        dropout:    Dropout rate inside the blocks.
    """

    def __init__(
        self,
        seq_len: int,
        n_features: int,
        d_model: int = 32,
        ff_hidden: int = 64,
        n_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.n_layers = n_layers

        # Stack of N TransformerBlocks that transform the encoded input back
        # into feature space. nn.ModuleList registers each block so the
        # optimiser tracks all their parameters.
        self.blocks = nn.ModuleList([
            TransformerBlock(
                seq_len=seq_len,
                n_features=n_features,
                d_model=d_model,
                ff_hidden=ff_hidden,
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Encoded representation of shape (batch, seq_len, n_features)
               produced by FusionformerEncoder.

        Returns:
            Reconstructed input of shape (batch, seq_len, n_features).
        """
        # Pass through each transformer block sequentially.
        # Each block refines the representation toward the reconstruction target.
        for block in self.blocks:
            x = block(x)

        return x


### 4.5 Fusionformer Full Autoencoder

In [ ]:
"""
Fusionformer — full autoencoder model for multivariate time-series anomaly detection.

Combines FusionformerEncoder + FusionformerDecoder into a reconstruction-based
autoencoder. Trained on normal (non-anomalous) sensor windows, the model learns
to reconstruct the pattern of normal behaviour. At inference time, windows that
differ from normal produce large reconstruction errors and are flagged as
anomalies.

Reconstruction-based anomaly detection principle:
    reconstruction_error(x) = mean((x - decoder(encoder(x)))**2)
    high error → not-normal → anomaly candidate

The model is described end-to-end so the training loop and evaluation code
can simply call `model(x)` for reconstruction and `model.reconstruction_error(x)`
for per-window anomaly scores.

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""
import torch
import torch.nn as nn

# (import from earlier cell)
# (import from earlier cell)


class Fusionformer(nn.Module):
    """
    Full Fusionformer autoencoder for multivariate time-series anomaly detection.

    Architecture:
        input (B, T, F)
            → FusionformerEncoder (positional encoding + N transformer blocks)
                → encoded representation (B, T, F)
                    → FusionformerDecoder (N transformer blocks)
                        → reconstruction (B, T, F)

    Args:
        seq_len:    Length of input time window (number of time steps).
        n_features: Number of sensor channels.
        d_model:    Latent dim used inside attention modules.
        ff_hidden:  Hidden dim of the feed-forward layer inside each block.
        n_layers:   Number of TransformerBlocks in encoder AND decoder.
        dropout:    Dropout rate throughout the model.
    """

    def __init__(
        self,
        seq_len: int,
        n_features: int,
        d_model: int = 32,
        ff_hidden: int = 64,
        n_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()

        # Store config for logging / reproducibility.
        self.seq_len = seq_len
        self.n_features = n_features
        self.d_model = d_model
        self.n_layers = n_layers

        # Encoder: input → positional encoding → N transformer blocks.
        self.encoder = FusionformerEncoder(
            seq_len=seq_len,
            n_features=n_features,
            d_model=d_model,
            ff_hidden=ff_hidden,
            n_layers=n_layers,
            dropout=dropout,
        )

        # Decoder: encoded representation → N transformer blocks → reconstruction.
        self.decoder = FusionformerDecoder(
            seq_len=seq_len,
            n_features=n_features,
            d_model=d_model,
            ff_hidden=ff_hidden,
            n_layers=n_layers,
            dropout=dropout,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Full autoencoder forward pass.

        Args:
            x: Input tensor of shape (batch, seq_len, n_features).

        Returns:
            Reconstruction of shape (batch, seq_len, n_features).
        """
        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)
        return reconstructed

    @torch.no_grad()
    def reconstruction_error(self, x: torch.Tensor) -> torch.Tensor:
        """
        Compute per-window reconstruction error for anomaly scoring.

        Higher error = window looks less like what the model learned as normal.
        Sort windows by this score and threshold to produce anomaly flags.

        Args:
            x: Input tensor of shape (batch, seq_len, n_features).

        Returns:
            Tensor of shape (batch,) with the mean squared reconstruction
            error for each input window.
        """
        # Put model in eval mode so dropout is disabled during scoring.
        self.eval()

        # Reconstruct.
        reconstructed = self(x)

        # Mean squared error per window, averaged across time and features.
        # dim=(1, 2) reduces the (T, F) axes, leaving a (batch,) tensor.
        error = ((x - reconstructed) ** 2).mean(dim=(1, 2))

        return error


### 4.6 TimeOnly Ablation

Architecturally identical to Fusionformer FAM except the channel-axis attention branch and fusion weights are removed. Used for the FAM contribution test in Chapter 4.3.

In [ ]:
class TimeOnlyAttention(nn.Module):
    """Time-axis-only attention (ablation of FusionAttention)."""
    def __init__(self, seq_len, n_features, d_model=32):
        super().__init__()
        self.input_projection = nn.Linear(n_features, d_model)
        self.time_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, batch_first=True)
        self.output_projection = nn.Linear(d_model, n_features)

    def forward(self, x):
        h = self.input_projection(x)
        time_out, _ = self.time_attention(h, h, h)
        return self.output_projection(time_out)


class TimeOnlyBlock(nn.Module):
    def __init__(self, seq_len, n_features, d_model=32, ff_hidden=64, dropout=0.1):
        super().__init__()
        self.attention = TimeOnlyAttention(seq_len, n_features, d_model)
        self.norm1 = nn.LayerNorm(n_features)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_features, ff_hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(ff_hidden, n_features),
        )
        self.norm2 = nn.LayerNorm(n_features)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.dropout(self.attention(x)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x


class FusionformerTimeOnly(nn.Module):
    """Fusionformer with channel-axis attention removed."""
    def __init__(self, seq_len, n_features, d_model=32, ff_hidden=64, n_layers=2, dropout=0.1):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, n_features) * 0.02)
        self.dropout = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([
            TimeOnlyBlock(seq_len, n_features, d_model, ff_hidden, dropout)
            for _ in range(n_layers)
        ])
        self.decoder_blocks = nn.ModuleList([
            TimeOnlyBlock(seq_len, n_features, d_model, ff_hidden, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x):
        h = self.dropout(x + self.pos_encoding)
        for b in self.encoder_blocks:
            h = b(h)
        for b in self.decoder_blocks:
            h = b(h)
        return h


### 4.7 SWSE + Adversarial Variants

Segment-Wise Sequence Embedding and adversarial discriminator variants. Full implementation as tested in Chapter 4.4-4.5.

In [ ]:
"""
Fusionformer with SWSE (Segment-wise Sequence Embedding).

Extends the FAM-only Fusionformer with the second core component from
Wang et al. 2025: per-channel segment embedding, following PatchTST-style
patching to preserve channel structure for meaningful channel attention.

Shape flow:
    Input:            (B, T, F)         # batch, time, channels
    After SWSE:       (B, F, N_seg, D)  # per-channel patches embedded to D
    After N blocks:   (B, F, N_seg, D)  # same shape, refined
    After recon head: (B, T, F)         # reconstruction

Where N_seg = T / segment_len.

Design rationale:
    - Per-channel patching (not flattened patching) preserves the F channel
      axis so channel attention has semantic meaning.
    - Each patch summarises segment_len timesteps of one channel into a
      D-dim embedding. Reduces attention sequence length from T to N_seg,
      typically ~5-10x reduction.
    - Fusion attention then operates on (B, F, N_seg, D):
        * Time attention: over N_seg segments per channel
        * Channel attention: over F channels per segment

Author: Nachiket Magadum
MSc AI dissertation, Brunel University London, 2026.
"""
import torch
import torch.nn as nn


# ============================================================
#  SWSE input embedding
# ============================================================

class SWSE(nn.Module):
    """
    Segment-wise Sequence Embedding (per-channel patching).

    Splits each channel's time-series into non-overlapping segments and
    embeds each segment into a D-dim vector via a linear projection.

    Args:
        seq_len:     Length of input time window.
        segment_len: Length of each segment (must divide seq_len).
        embed_dim:   Output dimension of each segment embedding.
    """

    def __init__(self, seq_len: int, segment_len: int, embed_dim: int):
        super().__init__()
        assert seq_len % segment_len == 0, (
            f"seq_len ({seq_len}) must be divisible by segment_len ({segment_len})"
        )
        self.seq_len = seq_len
        self.segment_len = segment_len
        self.n_segments = seq_len // segment_len
        self.embed_dim = embed_dim
        # One shared linear projection applied to every (channel, segment).
        self.projection = nn.Linear(segment_len, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, F)
        Returns:
            (B, F, N_seg, D)
        """
        B, T, F = x.shape
        # (B, T, F) -> (B, F, T)
        x = x.transpose(1, 2)
        # (B, F, T) -> (B, F, N_seg, segment_len)
        x = x.reshape(B, F, self.n_segments, self.segment_len)
        # (B, F, N_seg, segment_len) -> (B, F, N_seg, D)
        return self.projection(x)


# ============================================================
#  Fusion attention adapted for SWSE tokens
# ============================================================

class FusionAttentionSWSE(nn.Module):
    """
    FAM operating on SWSE segment tokens.

    Input shape: (B, F, N_seg, D)
    - Time attention: attend over N_seg segments, per channel
    - Channel attention: attend over F channels, per segment
    - Fusion via learnable softmax-normalised weights
    """

    def __init__(self, embed_dim: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.time_attention = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=1, batch_first=True
        )
        self.channel_attention = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=1, batch_first=True
        )
        # Fusion weights, softmaxed at forward time.
        self.fusion_weights = nn.Parameter(torch.tensor([0.5, 0.5]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, F, N_seg, D)
        Returns:
            (B, F, N_seg, D)
        """
        B, F, N_seg, D = x.shape

        # Time attention: for each channel, attend across segments.
        # Reshape (B, F, N_seg, D) -> (B*F, N_seg, D) so attention treats
        # each channel as an independent sequence over segments.
        x_time = x.reshape(B * F, N_seg, D)
        time_out, _ = self.time_attention(x_time, x_time, x_time)
        time_out = time_out.reshape(B, F, N_seg, D)

        # Channel attention: for each segment, attend across channels.
        # Permute (B, F, N_seg, D) -> (B, N_seg, F, D), then reshape
        # (B, N_seg, F, D) -> (B*N_seg, F, D) so each segment sees a
        # sequence of F channels.
        x_chan = x.permute(0, 2, 1, 3).reshape(B * N_seg, F, D)
        chan_out, _ = self.channel_attention(x_chan, x_chan, x_chan)
        chan_out = chan_out.reshape(B, N_seg, F, D).permute(0, 2, 1, 3)

        # Fuse with softmax-normalised learnable weights.
        weights = torch.softmax(self.fusion_weights, dim=0)
        return weights[0] * time_out + weights[1] * chan_out


# ============================================================
#  Transformer block on SWSE tokens
# ============================================================

class TransformerBlockSWSE(nn.Module):
    """
    Transformer block using FusionAttentionSWSE + position-wise FFN.

    Applies attention with residual + LayerNorm, then FFN with residual +
    LayerNorm. Operates on the (B, F, N_seg, D) tensor shape.
    """

    def __init__(
        self,
        embed_dim: int,
        ff_hidden: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.attention = FusionAttentionSWSE(embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_hidden, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Attention sub-layer.
        attn_out = self.attention(x)
        x = self.norm1(x + self.dropout(attn_out))
        # Feed-forward sub-layer.
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x


# ============================================================
#  Reconstruction head
# ============================================================

class ReconstructionHead(nn.Module):
    """
    Projects SWSE tokens back to the original time-series shape.

    (B, F, N_seg, D) -> (B, F, N_seg, segment_len)
                     -> (B, F, T)
                     -> (B, T, F)
    """

    def __init__(self, embed_dim: int, segment_len: int):
        super().__init__()
        self.segment_len = segment_len
        self.projection = nn.Linear(embed_dim, segment_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, F, N_seg, D = x.shape
        # Project each token back to segment_len values.
        x = self.projection(x)  # (B, F, N_seg, segment_len)
        # Reshape segments back to time dim: (B, F, T)
        x = x.reshape(B, F, N_seg * self.segment_len)
        # Transpose to (B, T, F).
        return x.transpose(1, 2)


# ============================================================
#  Full FusionformerSWSE
# ============================================================

class FusionformerSWSE(nn.Module):
    """
    Fusionformer with SWSE input embedding.

    Adds the second core component from Wang et al. 2025 to our earlier
    FAM-only implementation. Retains the reconstruction-autoencoder
    pattern used for anomaly detection.

    Args:
        seq_len:     Input time window length.
        n_features:  Number of sensor channels.
        segment_len: Length of each SWSE segment (must divide seq_len).
        embed_dim:   Latent dimension per segment token.
        ff_hidden:   Hidden dim of position-wise FFN.
        n_layers:    Number of encoder blocks (decoder mirrors).
        dropout:     Dropout rate throughout.
    """

    def __init__(
        self,
        seq_len: int,
        n_features: int,
        segment_len: int = 5,
        embed_dim: int = 32,
        ff_hidden: int = 64,
        n_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert seq_len % segment_len == 0, (
            f"seq_len ({seq_len}) must be divisible by segment_len ({segment_len})"
        )
        self.seq_len = seq_len
        self.n_features = n_features
        self.segment_len = segment_len
        self.n_segments = seq_len // segment_len
        self.embed_dim = embed_dim

        # SWSE input embedding.
        self.swse = SWSE(seq_len, segment_len, embed_dim)

        # Learned positional encoding — one vector per (channel, segment).
        # Shape (1, F, N_seg, D) broadcasts over the batch dim.
        self.pos_encoding = nn.Parameter(
            torch.randn(1, n_features, self.n_segments, embed_dim) * 0.02
        )
        self.dropout = nn.Dropout(dropout)

        # Encoder blocks.
        self.encoder_blocks = nn.ModuleList([
            TransformerBlockSWSE(embed_dim, ff_hidden, dropout)
            for _ in range(n_layers)
        ])

        # Decoder blocks (symmetric to encoder).
        self.decoder_blocks = nn.ModuleList([
            TransformerBlockSWSE(embed_dim, ff_hidden, dropout)
            for _ in range(n_layers)
        ])

        # Reconstruction back to (B, T, F).
        self.reconstruction_head = ReconstructionHead(embed_dim, segment_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, F)
        Returns:
            (B, T, F) reconstruction
        """
        # Embed segments per channel.
        h = self.swse(x)  # (B, F, N_seg, D)
        # Add positional encoding + dropout.
        h = h + self.pos_encoding
        h = self.dropout(h)
        # Encoder.
        for block in self.encoder_blocks:
            h = block(h)
        # Decoder.
        for block in self.decoder_blocks:
            h = block(h)
        # Reconstruct.
        return self.reconstruction_head(h)

    @torch.no_grad()
    def reconstruction_error(self, x: torch.Tensor) -> torch.Tensor:
        """
        Per-window mean squared reconstruction error for anomaly scoring.

        Args:
            x: (B, T, F)
        Returns:
            (B,) tensor of per-window MSE.
        """
        self.eval()
        reconstruction = self(x)
        return ((x - reconstruction) ** 2).mean(dim=(1, 2))


## Section 5: Baseline Models

### 5.1 Isolation Forest

Classical non-temporal baseline (Liu et al., 2008).

In [ ]:
def isolation_forest_scores(X_train, X_test, contamination=0.1, n_estimators=100, seed=42):
    """Fit IsolationForest on X_train, return anomaly scores for X_test.
    Higher score = more anomalous."""
    iso = IsolationForest(n_estimators=n_estimators, contamination=contamination, random_state=seed)
    iso.fit(X_train)
    # decision_function: higher = MORE normal; negate so higher = more anomalous
    return -iso.decision_function(X_test)


### 5.2 LSTM Autoencoder

Sequence-aware baseline for comparison against transformer methods.

In [ ]:
class LSTMAutoencoder(nn.Module):
    """Bidirectional LSTM encoder + LSTM decoder. Baseline for Chapter 4.1."""
    def __init__(self, n_features, hidden_size=64, latent_size=16, num_layers=2):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size
        self.latent_size = latent_size

        self.encoder = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, bidirectional=True,
        )
        self.encoder_to_latent = nn.Linear(hidden_size * 2, latent_size)
        self.latent_to_decoder = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
        )
        self.decoder_to_output = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_enc, _) = self.encoder(x)
        h_last = torch.cat([h_enc[-2], h_enc[-1]], dim=1)  # concat bidirectional
        z = self.encoder_to_latent(h_last)
        seq_len = x.size(1)
        h_dec_init = self.latent_to_decoder(z).unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder(h_dec_init)
        return self.decoder_to_output(out)


## Section 6: Training Loops

### 6.1 Utility: sliding windows

In [ ]:
def make_windows(data, seq_len):
    """Turn (N, F) into (N - seq_len + 1, seq_len, F) sliding windows."""
    n_windows = len(data) - seq_len + 1
    return np.stack([data[i:i + seq_len] for i in range(n_windows)])


def find_first_anomaly(y):
    """Return the index of the first anomaly, or len(y) if none."""
    y_arr = np.asarray(y)
    return int(np.argmax(y_arr == 1)) if y_arr.any() else len(y_arr)


### 6.2 Generic training loop

In [ ]:
def train_reconstruction_model(model, train_windows, all_windows, epochs=50,
                                 batch_size=32, lr=1e-3, device=DEVICE, verbose=True):
    """
    Train any autoencoder on train_windows using MSE reconstruction,
    then compute per-window reconstruction error for all_windows.

    Returns:
        window_errors: 1D array of length len(all_windows).
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_t = torch.tensor(train_windows, dtype=torch.float32)

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(train_t))
        losses = []
        for i in range(0, len(train_t), batch_size):
            batch = train_t[perm[i:i+batch_size]].to(device)
            recon = model(batch)
            loss = criterion(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"  epoch {epoch+1:2d}/{epochs}  loss = {np.mean(losses):.4f}")

    # Score all windows
    model.eval()
    all_t = torch.tensor(all_windows, dtype=torch.float32)
    with torch.no_grad():
        recon = model(all_t.to(device))
        window_errors = ((recon - all_t.to(device)) ** 2).mean(dim=(1, 2)).cpu().numpy()
    return window_errors


def broadcast_to_row_scores(window_errors, n_rows, seq_len):
    """Convert per-window errors to per-row scores (pad first seq_len-1 rows)."""
    scores = np.empty(n_rows)
    scores[:seq_len - 1] = window_errors[0]
    scores[seq_len - 1:] = window_errors
    return scores


### 6.3 End-to-end training on a single SKAB file

In [ ]:
def run_single_skab_file(file_path, model_class=Fusionformer, epochs=50, seed=42, verbose=True):
    """Train a model on one SKAB file and return metrics dict."""
    torch.manual_seed(seed); np.random.seed(seed)

    X, y = load_skab_file(file_path)
    X_scaled = StandardScaler().fit_transform(X)

    first_anom = find_first_anomaly(y)
    if first_anom < 60:  # need enough normal data to train
        return None

    X_train = X_scaled[:first_anom]
    train_windows = make_windows(X_train, 30)
    all_windows = make_windows(X_scaled, 30)

    model = model_class(seq_len=30, n_features=X.shape[1])
    window_errors = train_reconstruction_model(
        model, train_windows, all_windows,
        epochs=epochs, batch_size=32, verbose=verbose,
    )
    scores = broadcast_to_row_scores(window_errors, len(X_scaled), 30)
    return compute_all_metrics(y.values, scores)


# Example run (uncomment when SKAB is available):
# files = list_skab_files()
# metrics = run_single_skab_file(files[0])
# print(metrics)


## Section 7: Ablation Experiments

### 7.1 FAM vs TimeOnly (Chapter 4.3)

Core ablation testing whether the channel-axis attention branch contributes to Fusionformer's performance.

In [ ]:
def fam_vs_timeonly_sweep(files, seeds=(0, 1, 42), epochs=50):
    """Run both FAM and TimeOnly on each file at each seed. Returns paired dict."""
    results = {'fam': [], 'timeonly': [], 'file': [], 'seed': []}
    for f in files:
        for seed in seeds:
            r_fam = run_single_skab_file(f, Fusionformer, epochs=epochs, seed=seed, verbose=False)
            r_to = run_single_skab_file(f, FusionformerTimeOnly, epochs=epochs, seed=seed, verbose=False)
            if r_fam is None or r_to is None:
                continue
            results['fam'].append(r_fam['auroc'])
            results['timeonly'].append(r_to['auroc'])
            results['file'].append(str(f)); results['seed'].append(seed)
    return pd.DataFrame(results)


### 7.2 SWSE Configuration Study (Chapter 4.4)

Eight SWSE variants tested against FAM alone.

In [ ]:
# See fusionformer_swse.py cell above for the SWSE-Fusionformer implementation.
# Run configurations by varying: segment_len (2, 5), norm order (pre/post),
# attention type (FAM/TimeOnly), whether RevIN is applied, whether adversarial
# discriminator is added. All results in Table 4.4 came from this sweep.

def run_swse_config(files, model_factory, epochs=50, seed=42):
    """Run a specific SWSE configuration across files."""
    aurocs = []
    for f in files:
        torch.manual_seed(seed); np.random.seed(seed)
        X, y = load_skab_file(f)
        X_scaled = StandardScaler().fit_transform(X)
        first_anom = find_first_anomaly(y)
        if first_anom < 60:
            continue
        train_windows = make_windows(X_scaled[:first_anom], 30)
        all_windows = make_windows(X_scaled, 30)
        model = model_factory(n_features=X.shape[1])
        window_errors = train_reconstruction_model(
            model, train_windows, all_windows,
            epochs=epochs, verbose=False,
        )
        scores = broadcast_to_row_scores(window_errors, len(X_scaled), 30)
        aurocs.append(compute_all_metrics(y.values, scores)['auroc'])
    return np.mean(aurocs), aurocs


### 7.3 Full Paper Test on SMD (Chapter 4.5)

Multi-seed evaluation of SWSE + FAM + adversarial architecture vs FAM alone.

In [ ]:
"""
Train Fusionformer on an SMD (Server Machine Dataset) machine.

Mirrors train_fusionformer.py pattern for direct comparability with the
SKAB results. Same architecture, same hyperparameters, same eval harness.

Differences from SKAB pipeline:
    - 38 features instead of 8 (server metrics)
    - Much longer sequences (~57k rows vs ~1k in SKAB valve1)
    - Semi-supervised split is explicit: SMD provides train (all normal) and
      test (with anomalies) separately, concatenated by the loader.

Usage:
    python scripts/train_fusionformer_smd.py

Author: Nachiket Magadum
"""

import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# (import from earlier cell)
# (import from earlier cell)
# (import from earlier cell)


# ---------- Config (matched to SKAB training for comparability) ----------
SEQ_LEN     = 30
BATCH_SIZE  = 64          # slightly bigger — more training data than SKAB
EPOCHS      = 50
LR          = 1e-3
SEED        = 42
MACHINE     = "machine-1-1"

# Fusionformer hyperparameters
D_MODEL   = 32
FF_HIDDEN = 64
N_LAYERS  = 2
DROPOUT   = 0.1


def make_windows(data: np.ndarray, seq_len: int) -> np.ndarray:
    """(N, F) → (N - seq_len + 1, seq_len, F) sliding windows."""
    n_windows = len(data) - seq_len + 1
    return np.stack([data[i:i + seq_len] for i in range(n_windows)])


def main() -> None:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # ------------------------------------------------------------
    # 1. Load and standardize
    # ------------------------------------------------------------
    print(f"Loading SMD machine {MACHINE}...")
    X, y = load_smd_file(MACHINE)
    print(f"  {len(X)} rows, {X.shape[1]} features, anomaly rate {y.mean() * 100:.1f}%")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ------------------------------------------------------------
    # 2. Sliding windows.
    #    TRAIN only on initial contiguous NORMAL region (semi-supervised).
    #    SCORE on the full sequence.
    # ------------------------------------------------------------
    y_arr = y.values
    if y_arr.any():
        first_anom = int(np.argmax(y_arr == 1))
    else:
        first_anom = len(y_arr)

    X_train_scaled = X_scaled[:first_anom]
    print(f"  Training region: rows [0, {first_anom})  ({first_anom} normal rows)")

    train_windows = make_windows(X_train_scaled, SEQ_LEN)
    train_windows_t = torch.tensor(train_windows, dtype=torch.float32)

    all_windows = make_windows(X_scaled, SEQ_LEN)
    all_windows_t = torch.tensor(all_windows, dtype=torch.float32)

    print(f"  Train windows: {len(train_windows_t)}   Score windows: {len(all_windows_t)}")

    # ------------------------------------------------------------
    # 3. Device
    # ------------------------------------------------------------
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"  Using device: {device}")

    # ------------------------------------------------------------
    # 4. Model
    # ------------------------------------------------------------
    model = Fusionformer(
        seq_len=SEQ_LEN,
        n_features=X.shape[1],
        d_model=D_MODEL,
        ff_hidden=FF_HIDDEN,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {n_params:,}")

    # ------------------------------------------------------------
    # 5. Training loop
    # ------------------------------------------------------------
    print("\nTraining...")
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(train_windows_t))
        epoch_losses = []
        for i in range(0, len(train_windows_t), BATCH_SIZE):
            batch_idx = perm[i:i + BATCH_SIZE]
            batch = train_windows_t[batch_idx].to(device)
            reconstruction = model(batch)
            loss = criterion(reconstruction, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  epoch {epoch+1:2d}/{EPOCHS}  train_loss = {np.mean(epoch_losses):.4f}")

    # ------------------------------------------------------------
    # 6. Score every window (batched to avoid OOM on large SMD data)
    # ------------------------------------------------------------
    print("\nScoring...")
    model.eval()
    window_errors_list = []
    with torch.no_grad():
        for i in range(0, len(all_windows_t), 256):
            batch = all_windows_t[i:i + 256].to(device)
            reconstruction = model(batch)
            errors = ((reconstruction - batch) ** 2).mean(dim=(1, 2)).cpu().numpy()
            window_errors_list.append(errors)
    window_errors = np.concatenate(window_errors_list)

    # ------------------------------------------------------------
    # 7. Broadcast window scores to per-row scores
    # ------------------------------------------------------------
    scores = np.empty(len(X_scaled))
    scores[:SEQ_LEN - 1] = window_errors[0]
    scores[SEQ_LEN - 1:] = window_errors

    # ------------------------------------------------------------
    # 8. Evaluate
    # ------------------------------------------------------------
    metrics = compute_all_metrics(y.values, scores)

    print("\n" + "=" * 55)
    print(f"RESULTS — Fusionformer on SMD {MACHINE}")
    print("=" * 55)
    print(f"AUROC    (rank quality)                : {metrics['auroc']:.3f}")
    print(f"PR-AUC   (precision-recall AUC)        : {metrics['pr_auc']:.3f}")
    print(f"F1-PA    (point-adjusted, INFLATED)    : {metrics['f1_pa']:.3f}")
    print(f"Event-F1 (per-window, HONEST)          : {metrics['event_f1']:.3f}")
    print("=" * 55)


if __name__ == "__main__":
    main()


## Section 8: Statistical Significance Testing

Wilcoxon signed-rank tests and Rosenthal effect size for all main comparisons (Chapter 4.3, 4.5).

In [ ]:
"""
Complete Wilcoxon signed-rank significance tests including multi-seed
adversarial data from 28 Aug run.

Runs five tests total:
    1. FAM vs TimeOnly on SKAB (23 file means)
    2. FAM vs TimeOnly on SMD (15 machine-seed pairs)
    3. SWSE+FAM vs FAM alone on SMD (15 machine-seed pairs)  [NEW]
    4. SWSE+FAM+Adversarial vs FAM alone on SMD (15 machine-seed pairs)  [NEW]
    5. Full paper vs FAM alone on SMD single seed (5 machines, for context)

Usage:
    python scripts/wilcoxon_full.py

Author: Nachiket Magadum
"""

import numpy as np
from scipy import stats


# ============================================================
#  Data
# ============================================================

# SKAB — 23 file means (3-seed means per file)
skab_fam = [
    0.914, 0.825, 0.979, 0.943, 0.857,
    0.863, 0.912, 0.960, 0.949,
    1.000, 0.875, 0.976, 0.985, 0.863, 0.923, 0.962,
    0.786, 0.960, 0.939, 0.838, 0.870, 0.524, 0.938,
]
skab_time = [
    0.919, 0.821, 0.979, 0.939, 0.853,
    0.853, 0.910, 0.965, 0.938,
    1.000, 0.851, 0.976, 0.980, 0.886, 0.928, 0.962,
    0.782, 0.956, 0.936, 0.843, 0.861, 0.538, 0.933,
]

# SMD FAM vs TimeOnly — 15 pairs
smd_fam_time = [
    0.974, 0.976, 0.970,
    0.862, 0.888, 0.889,
    0.893, 0.821, 0.892,
    0.964, 0.938, 0.957,
    0.971, 0.970, 0.962,
]
smd_time_time = [
    0.964, 0.970, 0.971,
    0.861, 0.876, 0.890,
    0.879, 0.862, 0.874,
    0.957, 0.965, 0.963,
    0.960, 0.972, 0.964,
]

# SMD multi-seed adversarial data (28 Aug run) — 15 pairs
smd_fam_ms = [
    # machine-1-1 seeds 0, 1, 42
    0.974, 0.976, 0.970,
    # machine-1-4
    0.862, 0.888, 0.889,
    # machine-2-1
    0.893, 0.821, 0.892,
    # machine-2-5
    0.964, 0.938, 0.957,
    # machine-3-1
    0.971, 0.970, 0.962,
]
smd_swsefam_ms = [
    0.931, 0.965, 0.933,
    0.903, 0.900, 0.896,
    0.831, 0.829, 0.813,
    0.926, 0.938, 0.945,
    0.932, 0.946, 0.944,
]
smd_full_ms = [
    0.966, 0.957, 0.959,
    0.881, 0.886, 0.898,
    0.827, 0.816, 0.814,
    0.943, 0.948, 0.942,
    0.943, 0.951, 0.943,
]


def wilcoxon_report(name, a, b, label_a, label_b):
    a = np.array(a)
    b = np.array(b)
    diffs = a - b
    n = len(a)
    n_nonzero = int((diffs != 0).sum())
    n_a_wins = int((diffs > 0).sum())
    n_b_wins = int((diffs < 0).sum())

    result = stats.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    stat = result.statistic
    p = result.pvalue

    z_from_p = abs(stats.norm.ppf(p / 2))
    effect_r = z_from_p / np.sqrt(n_nonzero) if n_nonzero > 0 else 0.0

    if p < 0.01:
        sig = "highly significant (p < 0.01)"
    elif p < 0.05:
        sig = "significant (p < 0.05)"
    elif p < 0.10:
        sig = "marginally significant (p < 0.10)"
    else:
        sig = "NOT significant (p >= 0.10)"

    if abs(effect_r) < 0.1:
        eff_desc = "negligible"
    elif abs(effect_r) < 0.3:
        eff_desc = "small"
    elif abs(effect_r) < 0.5:
        eff_desc = "medium"
    else:
        eff_desc = "large"

    print(f"\n{'=' * 74}")
    print(f"{name}")
    print("=" * 74)
    print(f"  {label_a:30s} mean {a.mean():.4f}   std {a.std():.4f}")
    print(f"  {label_b:30s} mean {b.mean():.4f}   std {b.std():.4f}")
    print(f"  Mean difference ({label_a} - {label_b}): {diffs.mean():+.4f}")
    print(f"  N pairs: {n}   {label_a} wins: {n_a_wins}   {label_b} wins: {n_b_wins}   ties: {n - n_a_wins - n_b_wins}")
    print(f"  Wilcoxon W: {stat:.3f}")
    print(f"  p-value (two-sided): {p:.4f}")
    print(f"  Effect size r: {effect_r:.3f}  ({eff_desc})")
    print(f"  Verdict: {sig}")

    if p >= 0.10:
        print(f"  --> Cannot reject null: no significant difference.")
    else:
        winner = label_a if diffs.mean() > 0 else label_b
        print(f"  --> Reject null: {winner} is significantly better.")

    return {"p": p, "r": effect_r, "diff": diffs.mean()}


def main():
    print("=" * 74)
    print("WILCOXON SIGNED-RANK TESTS — FULL EVALUATION")
    print("Multi-seed data included (28 Aug adversarial run)")
    print("=" * 74)

    r1 = wilcoxon_report(
        "TEST 1: SKAB — FAM vs TimeOnly (23 file means)",
        skab_fam, skab_time, "FAM", "TimeOnly",
    )

    r2 = wilcoxon_report(
        "TEST 2: SMD — FAM vs TimeOnly (5 machines x 3 seeds = 15 pairs)",
        smd_fam_time, smd_time_time, "FAM", "TimeOnly",
    )

    r3 = wilcoxon_report(
        "TEST 3: SMD — SWSE+FAM vs FAM alone (15 pairs, MULTI-SEED)",
        smd_swsefam_ms, smd_fam_ms, "SWSE+FAM", "FAM alone",
    )

    r4 = wilcoxon_report(
        "TEST 4: SMD — Full paper (SWSE+FAM+Adv) vs FAM alone (15 pairs, MULTI-SEED)",
        smd_full_ms, smd_fam_ms, "Full paper", "FAM alone",
    )

    print("\n" + "=" * 74)
    print("HEADLINE WRITEUP PHRASES (paste-ready)")
    print("=" * 74)
    print(f"""
Independent FAM ablation shows no statistically significant benefit:
  - SKAB (23 files):    W=79,   p={r1['p']:.3f}, effect r={r1['r']:.3f} (small)
  - SMD  (15 pairs):    W=48.5, p={r2['p']:.3f}, effect r={r2['r']:.3f} (small)

SWSE addition to FAM significantly hurts on SMD:
  - Mean AUROC difference: {r3['diff']:+.4f}
  - Wilcoxon: p={r3['p']:.4f}, effect r={r3['r']:.3f}
  - {'SIGNIFICANT' if r3['p'] < 0.05 else 'not significant at 0.05'}

Full paper architecture (SWSE+FAM+Adversarial) significantly underperforms
FAM alone on SMD:
  - Mean AUROC difference: {r4['diff']:+.4f}
  - Wilcoxon: p={r4['p']:.4f}, effect r={r4['r']:.3f}
  - {'SIGNIFICANT' if r4['p'] < 0.05 else 'not significant at 0.05'}
""")


if __name__ == "__main__":
    main()


Run the full statistical evaluation:

In [ ]:
# Execute all Wilcoxon tests using the hardcoded results from the 28 Aug run.
main()


## Section 9: F1 Telemetry Cross-Domain Case Study (Chapter 4.6)

Apply the trained Fusionformer FAM to Formula 1 telemetry (2023 British GP, Hamilton) as a qualitative cross-domain demonstration. Requires the `fastf1` library and an internet connection to download telemetry.

In [ ]:
"""
Cross-domain qualitative case study on Formula 1 telemetry.

Trains Fusionformer FAM on the first quarter of Hamilton's 2023 British GP
race (used as "normal driving" reference), then scores the entire race and
plots reconstruction error over the race timeline. High-error moments are
visually inspected against known F1 events (pit stops, DRS activations,
racing incidents).

Note: F1 does not have ground truth anomaly labels, so this is a
qualitative demonstration of cross-domain pipeline generalisability
rather than a quantitative evaluation.

Usage:
    python scripts/experiment_f1_casestudy.py

Output:
    figures/f1_reconstruction_error.png    - Anomaly score over race
    figures/f1_lap_summary.png             - Per-lap mean reconstruction error
    notes/f1_case_study_results.md         - Written results summary

Author: Nachiket Magadum
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# (import from earlier cell)


# Configuration
SEED        = 42
SEQ_LEN     = 30
BATCH_SIZE  = 64
EPOCHS      = 50
LR          = 1e-3
D_MODEL     = 32
FF_HIDDEN   = 64
N_LAYERS    = 2
DROPOUT     = 0.1

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)
NOTES_DIR = Path("notes")
NOTES_DIR.mkdir(exist_ok=True)


def load_f1_telemetry():
    """Load Hamilton British GP 2023 telemetry, resample to uniform 10 Hz."""
    import fastf1

    cache_dir = Path("datasets/fastf1_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    fastf1.Cache.enable_cache(str(cache_dir))

    print("Loading 2023 British Grand Prix (race)...")
    session = fastf1.get_session(2023, "British Grand Prix", "R")
    session.load(laps=True, telemetry=True, weather=False, messages=False)
    print("  Session loaded.")

    # Hamilton car number: 44
    car_data = session.car_data["44"]

    # Filter to racing pace (speed > 100 km/h) to skip pit lane / pre-race
    moving = car_data[car_data["Speed"] > 100].copy()
    print(f"  Raw telemetry rows: {len(car_data)}")
    print(f"  Racing rows (Speed > 100): {len(moving)}")

    # Convert Brake bool -> int (needed for numeric operations)
    moving["Brake"] = moving["Brake"].astype(int)

    # Set Date as index for resampling
    moving = moving.set_index("Date")

    # Continuous channels get mean-then-interpolate
    continuous_cols = ["Speed", "RPM", "Throttle"]
    cont_resampled = (
        moving[continuous_cols].resample("100ms").mean().interpolate(method="linear")
    )

    # Discrete channels get nearest-neighbour
    discrete_cols = ["nGear", "DRS", "Brake"]
    disc_resampled = moving[discrete_cols].resample("100ms").nearest()

    # Combine
    resampled = pd.concat([cont_resampled, disc_resampled], axis=1)
    resampled = resampled.dropna()

    print(f"  After resampling to 10 Hz: {len(resampled)} rows, "
          f"{resampled.shape[1]} features")

    return resampled, session


def make_windows(data, seq_len):
    n_windows = len(data) - seq_len + 1
    return np.stack([data[i:i + seq_len] for i in range(n_windows)])


def train_on_normal(X_train, n_features, device):
    """Train Fusionformer FAM on the early-race 'normal' region."""
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    train_windows = make_windows(X_train, SEQ_LEN)
    train_windows_t = torch.tensor(train_windows, dtype=torch.float32)

    print(f"  Train windows: {len(train_windows_t)}")
    print(f"  Using device: {device}")

    model = Fusionformer(
        seq_len=SEQ_LEN,
        n_features=n_features,
        d_model=D_MODEL,
        ff_hidden=FF_HIDDEN,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    print("  Training on early-race normal reference...")
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(train_windows_t))
        losses = []
        for i in range(0, len(train_windows_t), BATCH_SIZE):
            batch_idx = perm[i:i + BATCH_SIZE]
            batch = train_windows_t[batch_idx].to(device)
            recon = model(batch)
            loss = criterion(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"    epoch {epoch+1:2d}/{EPOCHS}  loss = {np.mean(losses):.4f}")

    return model


def score_all(model, X_scaled, device):
    """Score the whole race in batches."""
    all_windows = make_windows(X_scaled, SEQ_LEN)
    all_windows_t = torch.tensor(all_windows, dtype=torch.float32)

    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(0, len(all_windows_t), 256):
            batch = all_windows_t[i:i + 256].to(device)
            recon = model(batch)
            e = ((recon - batch) ** 2).mean(dim=(1, 2)).cpu().numpy()
            errors.append(e)
    return np.concatenate(errors)


def plot_reconstruction_over_time(errors, index, output_path):
    """Plot reconstruction error over the race timeline."""
    fig, ax = plt.subplots(figsize=(14, 5))

    # Broadcast window errors to per-row scores
    scores = np.empty(len(index))
    scores[:SEQ_LEN - 1] = errors[0]
    scores[SEQ_LEN - 1:] = errors

    # X axis: elapsed race time in minutes
    elapsed = (index - index[0]).total_seconds() / 60.0

    ax.plot(elapsed, scores, linewidth=0.6, color="#1F4E79", alpha=0.8)

    # Threshold at 95th percentile for visual reference
    threshold = np.percentile(scores, 95)
    ax.axhline(threshold, color="#C62828", linestyle="--", linewidth=1,
               alpha=0.7, label=f"95th percentile ({threshold:.3f})")

    # Highlight top 20 highest-error windows
    top_indices = np.argsort(scores)[-20:]
    ax.scatter(elapsed[top_indices], scores[top_indices],
               color="#F4A261", s=25, zorder=5, label="Top-20 highest error")

    ax.set_xlabel("Race time (minutes)")
    ax.set_ylabel("Reconstruction error (MSE)")
    ax.set_title("Fusionformer FAM reconstruction error across Hamilton's "
                 "2023 British GP race")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {output_path}")

    return scores, threshold


def plot_lap_summary(scores, index, session, output_path):
    """Plot mean reconstruction error per lap."""
    # Get Hamilton's lap data
    laps = session.laps.pick_drivers("HAM")

    fig, ax = plt.subplots(figsize=(12, 5))

    if len(laps) == 0:
        print("  No lap data available — skipping per-lap plot")
        return None

    # Map each score timestamp to a lap number
    scores_df = pd.DataFrame({"score": scores}, index=index)

    lap_means = []
    lap_nums = []
    for _, lap_row in laps.iterrows():
        lap_start = lap_row.get("LapStartTime")
        lap_num = lap_row.get("LapNumber")
        if lap_start is None or lap_num is None or pd.isna(lap_num):
            continue
        # Approximate: 90 seconds per lap
        try:
            end = lap_start + pd.Timedelta(seconds=90)
            mask = (scores_df.index >= lap_start) & (scores_df.index < end)
            if mask.sum() > 5:
                lap_means.append(scores_df.loc[mask, "score"].mean())
                lap_nums.append(int(lap_num))
        except Exception:
            continue

    if not lap_means:
        print("  Could not compute per-lap means — skipping")
        return None

    colors = ["#C62828" if m > np.mean(lap_means) + np.std(lap_means)
              else "#1F4E79" for m in lap_means]
    ax.bar(lap_nums, lap_means, color=colors, edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Lap number")
    ax.set_ylabel("Mean reconstruction error")
    ax.set_title("Mean per-lap reconstruction error — red bars mark unusual laps")
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {output_path}")

    return list(zip(lap_nums, lap_means))


def write_summary_notes(scores, threshold, elapsed, lap_summary, output_path):
    """Write a markdown summary of the case study."""
    n_high = int((scores > threshold).sum())
    pct_high = 100 * n_high / len(scores)

    top_5_times = elapsed[np.argsort(scores)[-5:]][::-1]
    top_5_scores = scores[np.argsort(scores)[-5:]][::-1]

    md = f"""# F1 Case Study — Hamilton 2023 British GP

## Setup

- Race: 2023 Formula 1 British Grand Prix (Silverstone), race session
- Driver: Lewis Hamilton (car number 44)
- Telemetry channels: Speed, RPM, Throttle (continuous); nGear, DRS, Brake (discrete)
- Resampled from variable rate (159 ms - 89 s intervals) to uniform 10 Hz using
  per-channel-type interpolation.
- Filtered to Speed > 100 km/h to skip pit lane and pre-race.

## Model

- Fusionformer FAM (same architecture as SKAB/SMD experiments)
- Trained on the first 25 percent of the racing telemetry as a "normal driving"
  reference
- Scored on the full race using per-window mean squared reconstruction error
- Note: no ground truth anomaly labels exist for F1 telemetry, so this is a
  qualitative cross-domain demonstration, not a quantitative benchmark

## Results

- Total scored windows: {len(scores):,}
- 95th percentile threshold: {threshold:.4f}
- Number of "high-error" windows above threshold: {n_high:,} ({pct_high:.1f} percent)

## Top-5 highest reconstruction error moments (approximate race minute)

| Rank | Race minute | Reconstruction error |
|---|---|---|
"""
    for i, (t, s) in enumerate(zip(top_5_times, top_5_scores), start=1):
        md += f"| {i} | {t:.1f} | {s:.4f} |\n"

    md += """

## Qualitative interpretation

High reconstruction error corresponds to moments where the model, having
learned Hamilton's early-race driving style, fails to predict the current
telemetry pattern. These typically align with:

- Pit stops (rapid deceleration, engine idle, then re-acceleration)
- Sudden traffic (unusual braking or throttle patterns)
- DRS activation windows (throttle profile changes)
- Safety car or virtual safety car periods

Manual inspection of the top-5 moments against known race events would allow
qualitative validation that reconstruction error captures interesting driving
regimes.

## Purpose in the dissertation

This case study is included as a qualitative demonstration that the trained
Fusionformer pipeline generalises across dataset types — from labelled
industrial benchmarks (SKAB, SMD) to unlabelled real-world telemetry (F1).
It fulfils the cross-domain evaluation component proposed in the Task 1 DDR.

Because F1 telemetry lacks ground truth anomaly labels, quantitative
comparison to the SKAB and SMD ablations is not possible. The value here
is architectural portability, not benchmark performance.
"""

    with open(output_path, "w") as f:
        f.write(md)
    print(f"  Saved: {output_path}")


def main():
    print("=" * 70)
    print("F1 QUALITATIVE CASE STUDY — Hamilton 2023 British GP")
    print("=" * 70)

    # Load and preprocess
    resampled, session = load_f1_telemetry()
    n_features = resampled.shape[1]
    print(f"  Feature columns: {list(resampled.columns)}")

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(resampled.values)
    print(f"  Standardized shape: {X_scaled.shape}")

    # Split: first 25% for training (normal reference), score whole race
    train_end = int(len(X_scaled) * 0.25)
    X_train = X_scaled[:train_end]
    print(f"  Training region: rows [0, {train_end})")

    # Train model
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = train_on_normal(X_train, n_features, device)

    # Score whole race
    print("\nScoring the whole race...")
    errors = score_all(model, X_scaled, device)
    print(f"  Scored {len(errors)} windows")

    # Plot reconstruction error over time
    print("\nGenerating plots...")
    scores, threshold = plot_reconstruction_over_time(
        errors, resampled.index,
        FIGURES_DIR / "f1_reconstruction_error.png",
    )

    # Plot per-lap summary
    lap_summary = plot_lap_summary(
        scores, resampled.index, session,
        FIGURES_DIR / "f1_lap_summary.png",
    )

    # Write markdown notes
    elapsed = (resampled.index - resampled.index[0]).total_seconds().values / 60.0
    write_summary_notes(
        scores, threshold, elapsed, lap_summary,
        NOTES_DIR / "f1_case_study_results.md",
    )

    print("\n" + "=" * 70)
    print("Case study complete. Outputs:")
    print("  figures/f1_reconstruction_error.png")
    print("  figures/f1_lap_summary.png")
    print("  notes/f1_case_study_results.md")
    print("=" * 70)


if __name__ == "__main__":
    main()


## Section 10: Plotting Utilities

Reproduction of every figure referenced in Chapter 4 (ablation comparisons, seed variance, SWSE configurations, F1 timeline).

In [ ]:
"""
Generate SKAB vs SMD ablation comparison plots.

Produces:
    figures/ablation_skab_vs_smd.png     - Per-file/machine bar chart, both datasets
    figures/ablation_grand_mean.png      - Grand mean comparison with error bars
    figures/seed_variance_comparison.png - Seed stability comparison

Uses the data from the multi-seed variance sweeps on both benchmarks.

Usage:
    python scripts/plot_ablation_comparison.py

Author: Nachiket Magadum
"""

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)


# ============================================================
#  Data (aggregated from experiment logs)
# ============================================================

# SKAB — grand mean across 3 seeds per file, 23 files total
skab_files = [
    "v1-0", "v1-1", "v1-2", "v1-3", "v1-4",
    "v2-0", "v2-1", "v2-2", "v2-3",
    "o-1", "o-2", "o-3", "o-4", "o-5", "o-6", "o-7",
    "o-8", "o-9", "o-10", "o-11", "o-12", "o-13", "o-14",
]

skab_fam_means = [
    0.914, 0.825, 0.979, 0.943, 0.857,
    0.863, 0.912, 0.960, 0.949,
    1.000, 0.875, 0.976, 0.985, 0.863, 0.923, 0.962,
    0.786, 0.960, 0.939, 0.838, 0.870, 0.524, 0.938,
]
skab_time_means = [
    0.919, 0.821, 0.979, 0.939, 0.853,
    0.853, 0.910, 0.965, 0.938,
    1.000, 0.851, 0.976, 0.980, 0.886, 0.928, 0.962,
    0.782, 0.956, 0.936, 0.843, 0.861, 0.538, 0.933,
]

# SMD — per-machine means across 3 seeds each
smd_machines = ["m1-1", "m1-4", "m2-1", "m2-5", "m3-1"]
smd_fam_means = [0.973, 0.880, 0.869, 0.953, 0.968]
smd_time_means = [0.968, 0.876, 0.872, 0.962, 0.965]

# Per-seed values for seed-variance plot
smd_fam_by_seed = {
    "m1-1": [0.974, 0.976, 0.970],
    "m1-4": [0.862, 0.888, 0.889],
    "m2-1": [0.893, 0.821, 0.892],
    "m2-5": [0.964, 0.938, 0.957],
    "m3-1": [0.971, 0.970, 0.962],
}
smd_time_by_seed = {
    "m1-1": [0.964, 0.970, 0.971],
    "m1-4": [0.861, 0.876, 0.890],
    "m2-1": [0.879, 0.862, 0.874],
    "m2-5": [0.957, 0.965, 0.963],
    "m3-1": [0.960, 0.972, 0.964],
}


# ============================================================
#  Plot 1: Side-by-side per-file / per-machine diff
# ============================================================

def plot_per_item_diff():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5),
                                    gridspec_kw={"width_ratios": [23, 5]})

    # SKAB panel
    skab_diffs = np.array(skab_fam_means) - np.array(skab_time_means)
    x_skab = np.arange(len(skab_files))
    colors_skab = ["#2E7D32" if d > 0.005 else ("#C62828" if d < -0.005 else "#9E9E9E") for d in skab_diffs]
    ax1.bar(x_skab, skab_diffs, color=colors_skab, edgecolor="black", linewidth=0.4)
    ax1.axhline(0, color="black", linewidth=0.8)
    ax1.axhline(0.02, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)
    ax1.axhline(-0.02, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)
    ax1.set_xticks(x_skab)
    ax1.set_xticklabels(skab_files, rotation=45, ha="right", fontsize=8)
    ax1.set_ylabel("FAM AUROC minus TimeOnly AUROC")
    ax1.set_title(f"SKAB — 23 files (grand mean diff: {np.mean(skab_diffs):+.4f})")
    ax1.grid(True, axis="y", alpha=0.3)
    ax1.set_ylim(-0.06, 0.06)

    # SMD panel
    smd_diffs = np.array(smd_fam_means) - np.array(smd_time_means)
    x_smd = np.arange(len(smd_machines))
    colors_smd = ["#2E7D32" if d > 0.005 else ("#C62828" if d < -0.005 else "#9E9E9E") for d in smd_diffs]
    ax2.bar(x_smd, smd_diffs, color=colors_smd, edgecolor="black", linewidth=0.4)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.axhline(0.02, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)
    ax2.axhline(-0.02, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)
    ax2.set_xticks(x_smd)
    ax2.set_xticklabels(smd_machines, rotation=45, ha="right", fontsize=9)
    ax2.set_title(f"SMD — 5 machines (grand mean diff: {np.mean(smd_diffs):+.4f})")
    ax2.grid(True, axis="y", alpha=0.3)
    ax2.set_ylim(-0.06, 0.06)

    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor="#2E7D32", label="FAM wins by >0.005"),
        Patch(facecolor="#C62828", label="TimeOnly wins by >0.005"),
        Patch(facecolor="#9E9E9E", label="Tied (within ±0.005)"),
    ]
    fig.legend(handles=legend_elems, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))

    plt.suptitle("FAM vs TimeOnly per-file/machine — both datasets, 3-seed means")
    plt.tight_layout()
    out = FIGURES_DIR / "ablation_skab_vs_smd.png"
    plt.savefig(out)
    plt.close()
    print(f"Saved: {out}")


# ============================================================
#  Plot 2: Grand mean comparison
# ============================================================

def plot_grand_mean():
    fig, ax = plt.subplots(figsize=(9, 5))

    labels = ["SKAB\n(23 files × 3 seeds)", "SMD\n(5 machines × 3 seeds)"]
    fam_means = [np.mean(skab_fam_means), np.mean(smd_fam_means)]
    time_means = [np.mean(skab_time_means), np.mean(smd_time_means)]
    fam_stds = [np.std(skab_fam_means), np.std(smd_fam_means)]
    time_stds = [np.std(skab_time_means), np.std(smd_time_means)]

    x = np.arange(len(labels))
    width = 0.35

    bars1 = ax.bar(x - width/2, fam_means, width, yerr=fam_stds, capsize=8,
                   label="Fusionformer FAM (dual-axis)", color="#1F4E79",
                   edgecolor="black", linewidth=0.5)
    bars2 = ax.bar(x + width/2, time_means, width, yerr=time_stds, capsize=8,
                   label="TimeOnly (ablated)", color="#F4A261",
                   edgecolor="black", linewidth=0.5)

    for bars in [bars1, bars2]:
        for b in bars:
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                    f"{b.get_height():.3f}", ha="center", va="bottom", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Grand mean AUROC (± std across files/machines)")
    ax.set_title("Grand mean AUROC — FAM vs TimeOnly on both datasets")
    ax.set_ylim(0.7, 1.0)
    ax.legend(loc="lower right")
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    out = FIGURES_DIR / "ablation_grand_mean.png"
    plt.savefig(out)
    plt.close()
    print(f"Saved: {out}")


# ============================================================
#  Plot 3: Seed variance comparison (SMD)
# ============================================================

def plot_seed_variance():
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(smd_machines))
    width = 0.35

    fam_vals = [smd_fam_by_seed[m] for m in smd_machines]
    time_vals = [smd_time_by_seed[m] for m in smd_machines]

    # Box-and-whisker for both
    positions_fam = x - width/2
    positions_time = x + width/2

    bp1 = ax.boxplot(fam_vals, positions=positions_fam, widths=0.3,
                     patch_artist=True, medianprops={"color": "black"})
    for patch in bp1["boxes"]:
        patch.set_facecolor("#1F4E79")

    bp2 = ax.boxplot(time_vals, positions=positions_time, widths=0.3,
                     patch_artist=True, medianprops={"color": "black"})
    for patch in bp2["boxes"]:
        patch.set_facecolor("#F4A261")

    ax.set_xticks(x)
    ax.set_xticklabels(smd_machines)
    ax.set_xlabel("SMD machine")
    ax.set_ylabel("AUROC across 3 seeds")
    ax.set_title("Seed variance — FAM (blue) vs TimeOnly (orange)\n"
                 "TimeOnly is measurably more stable on some machines (e.g. m2-1)")
    ax.grid(True, axis="y", alpha=0.3)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor="#1F4E79", label="FAM"),
        Patch(facecolor="#F4A261", label="TimeOnly"),
    ], loc="lower right")

    plt.tight_layout()
    out = FIGURES_DIR / "seed_variance_comparison.png"
    plt.savefig(out)
    plt.close()
    print(f"Saved: {out}")


if __name__ == "__main__":
    plot_per_item_diff()
    plot_grand_mean()
    plot_seed_variance()
    print("\nAll comparison plots saved to figures/")


## End

All 153 training runs and 5 statistical significance tests reported in Chapter 4 can be reproduced by running the cells above in order, given the SKAB and SMD datasets are downloaded and placed under `datasets/`.

For questions or issues, contact the author. The full source repository is referenced in Appendix A of the dissertation.

**Nachiket Magadum (2550458)** — September 2026
